In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense

data = pd.read_excel('base.xls')

# Se os dados estiverem em uma única coluna, você pode converter para um array numpy
data = data.values

# Normalizar os dados
scaler = MinMaxScaler(feature_range=(-1, 1))
data_normalized = scaler.fit_transform(data)

# Definir o número de passos no tempo para a sequência de entrada
n_steps = 10

# Dividir os dados em sequências de entrada e saída
X, y = [], []
for i in range(len(data_normalized) - n_steps):
    X.append(data_normalized[i:i + n_steps])
    y.append(data_normalized[i + n_steps])
X, y = np.array(X), np.array(y)

# Reshape para o formato (samples, time steps, features)
n_features = 1  # Número de features (nesse caso, apenas uma coluna)
X = X.reshape((X.shape[0], X.shape[1], n_features))

# Definir a arquitetura da rede LSTM
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(n_steps, n_features)))
model.add(Dense(1))  # Camada de saída com uma única unidade

# Compilar o modelo
model.compile(optimizer='adam', loss='mse')

# Treinar o modelo
model.fit(X, y, epochs=200, verbose=1)

# Avaliar o modelo - você pode usar conjuntos de validação e teste separados
# score = model.evaluate(X_test, y_test)

# Fazer previsões
input_data = data_normalized[-n_steps:]  # Últimos n_steps pontos de dados
input_data = input_data.reshape((1, n_steps, n_features))
predicted_output = model.predict(input_data)
predicted_output = scaler.inverse_transform(predicted_output)  # Desnormalizar os dados

print("Próximo valor previsto:", predicted_output[0][0])
